In [1]:

from claimbuster_spotter.adv_transformer.core.utils.flags import FLAGS


In [2]:
import os
# display current working directory
os.getcwd()

'/home/adamj/factcheck-podcasts/src'

In [3]:
FLAGS.cs_model_dir = "/home/adamj/factcheck-podcasts/src/claimbuster_spotter/output/bba/"

In [4]:
from claimbuster_spotter.adv_transformer.core.api.api_wrapper import ClaimSpotterAPI
claimspotter = ClaimSpotterAPI()

2023-04-13 15:59:26.083285: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
[nltk_data] Downloading package punkt to /home/adamj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/adamj/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /home/adamj/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package stopwords to /home/adamj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dependencies Loaded.


2023-04-13 15:59:28.221849: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2023-04-13 15:59:28.329750: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-04-13 15:59:28.329782: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 938.86GiB/s
2023-04-13 15:59:28.329796: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2023-04-13 15:59:28.331452: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2023-04-13 15:59:28.331530: I tensorflow/stream_execut

In [5]:
claimspotter.single_sentence_query("This is a test")

2023-04-13 15:59:32.348308: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2023-04-13 15:59:32.353902: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 3187200000 Hz


array([[0.72771011, 0.27228989]])

In [6]:
sentence_list = [
    'Donald Trump is the 45th President of the United States',
    'I really like cheese',
    'McDonalds earns $10 billion dollars each minute'
]

In [7]:
claimspotter.batch_sentence_query(sentence_list)

array([[0.64611771, 0.35388229],
       [0.86793979, 0.13206021],
       [0.07725842, 0.92274158]])

In [8]:
import requests

In [ ]:
podcasts = requests.get("http://127.0.0.1:8008/api/podcasts/")
podcasts = podcasts.json()

In [ ]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)


In [10]:
segmentation_uuids = ["a42c746c-d95d-11ed-a2ca-00155da78120"]

In [11]:
for seg_uuid in segmentation_uuids:
    segments = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
    segments = segments.json()
    sentence_list = [utt["text"] for utt in segments["utterance_set"]]
    scores = claimspotter.batch_sentence_query(sentence_list)

    for i, segment in enumerate(segments["utterance_set"]):
        requests.post(f"http://localhost:8008/api/classifications/{segment['uuid']}/", json={
            "utterance": segment["uuid"],
            "qualifier": "Checkworthiness",
            "category": "Checkworthy",
            "label": str(scores[i][1]),
            "agent": "ClaimBuster-BBA"
        })